# Línea base de otros modelos en Kaggle (IIC-3694)

Mide pass@1 y pass@5 sobre `eval` para modelos distintos del objetivo, con las mismas 90 instancias por condición y 5 muestras que la línea base de Qwen3-1.7B con thinking. No corre nada sobre `train` ni arma datos SFT/DPO. **Continúa una corrida anterior** si se la das como entrada.

- GPU 0, sin thinking (respuestas cortas, rápido): Llama-3.2-1B, Qwen3-1.7B y Qwen3-4B.
- GPU 1, con thinking (lento, unas 7 horas): Qwen3-0.6B.

Pasos (detalle en `docs/guia_kaggle.md`):

1. Crea un *dataset* de Kaggle con `inventory_harness_colab.zip` (sale de `python scripts/package_for_colab.py`).
2. En este cuaderno: *Add Input* con ese dataset; en *Settings*, **Accelerator: GPU T4 x2** e **Internet: On**.
3. *Save Version* → **Save & Run All (Commit)**. Corre en segundo plano, con el computador apagado, hasta 12 horas.
4. Al terminar, descarga `resultados_otros_modelos.zip` desde la pestaña *Output* de esa versión.


## 1. GPU

In [ ]:
import os
import time

INICIO = time.time()
PRUEBA_SECA = os.environ.get("PRUEBA_SECA") == "1"  # solo para probar el cuaderno sin GPU

if PRUEBA_SECA:
    N_GPU = int(os.environ.get("N_GPU", "2"))
else:
    import torch

    N_GPU = torch.cuda.device_count()
    for i in range(N_GPU):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
if N_GPU == 0:
    # Sin GPU correria en CPU hasta LIMITE_HORAS y gastaria la sesion sin avanzar.
    raise SystemExit("NO HAY GPU. Settings > Accelerator > GPU T4 x2, y vuelve a ejecutar.")

## 2. Encontrar el código y los resultados anteriores

Busca en `/kaggle/input` el zip del código (o la carpeta, si Kaggle lo descomprimió) y las carpetas `linea_base_sin_think` y `linea_base_qwen06b` de una corrida anterior. Las copia a `/kaggle/working`, que es lo que Kaggle guarda al final.

In [ ]:
import shutil
import sys
import zipfile
from pathlib import Path

ENTRADA = Path(os.environ.get("ENTRADA_KAGGLE", "/kaggle/input"))
SALIDA = Path(os.environ.get("SALIDA_KAGGLE", "/kaggle/working"))
CODIGO = Path(os.environ.get("CODIGO_KAGGLE", "/tmp/inventory_harness"))
RESULTADOS = SALIDA / "results"
RESULTADOS.mkdir(parents=True, exist_ok=True)
CARPETAS = ("linea_base_sin_think", "linea_base_qwen06b")


def buscar_codigo():
    for ruta in sorted(ENTRADA.glob("**/inventory_harness_colab.zip")):
        if ruta.is_file():
            return ruta
    for script in sorted(ENTRADA.glob("**/scripts/run_pilot.py")):
        raiz = script.parent.parent
        if (raiz / "src" / "inventory_harness").is_dir():
            return raiz
    return None


origen = buscar_codigo()
if origen is None:
    raise SystemExit("No encontre el codigo en /kaggle/input. Agrega el dataset con Add Input.")
shutil.rmtree(CODIGO, ignore_errors=True)
if origen.is_file():
    with zipfile.ZipFile(origen) as archivo:
        archivo.extractall(CODIGO)
else:
    shutil.copytree(origen, CODIGO)
sys.path.insert(0, str(CODIGO / "src"))
print("Codigo desde", origen)


def rollouts_hechos(carpeta):
    total = 0
    for archivo in carpeta.glob("rollouts_*.jsonl"):
        with archivo.open(encoding="utf-8", errors="replace") as f:
            total += sum(1 for linea in f if linea.strip())
    return total


for nombre in CARPETAS:
    destino = RESULTADOS / nombre
    previas = sorted(m.parent for m in ENTRADA.glob(f"**/{nombre}/manifest.json"))
    if destino.exists():
        print(f"{nombre}: ya esta en {destino} ({rollouts_hechos(destino)} rollouts); se continua.")
    elif previas:
        shutil.copytree(previas[0], destino)
        print(f"{nombre}: copiada desde {previas[0]} ({rollouts_hechos(destino)} rollouts); se continua.")
        if len(previas) > 1:
            print("  AVISO: habia varias; se uso la primera:", *previas, sep="\n  ")
    else:
        print(f"{nombre}: no hay corrida anterior; empieza de cero.")

## 3. Dependencias y pruebas del núcleo

In [ ]:
import subprocess

if not PRUEBA_SECA:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51", "accelerate", "pytest"],
        check=True,
    )

In [ ]:
prueba = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_solver.py",
     "tests/test_numbers_es.py", "tests/test_verifier.py", "tests/test_generator.py",
     "tests/test_tools.py"],
    cwd=CODIGO, capture_output=True, text=True,
)
print(prueba.stdout[-500:], prueba.stderr[-500:])
if prueba.returncode != 0:
    # Una version guardada corre sola: mejor detenerse que gastar horas con codigo roto.
    raise SystemExit("Fallaron las pruebas del nucleo: revisa el zip del codigo.")

## 4. Configuración

`POR_CELDA` y `MUESTRAS` **no se cambian**: son los de la línea base de Qwen3-1.7B con thinking, y con otros valores los modelos no se comparan sobre las mismas instancias. Las listas de modelos sí se pueden editar. `LIMITE_HORAS` detiene todo a tiempo para guardar antes del tope de 12 horas de Kaggle; lo que alcance a correr queda guardado y se puede continuar en otra versión.


In [ ]:
# Sin thinking el tope es de 768 tokens: cada modelo toma minutos, no horas.
MODELOS_SIN_THINK = ["unsloth/Llama-3.2-1B-Instruct", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]
BATCH_SIN_THINK = 16
# Con thinking genera hasta 4096 tokens por respuesta: es la tanda larga.
MODELOS_CON_THINK = ["Qwen/Qwen3-0.6B"]
BATCH_CON_THINK = 8  # con 16 no cabe en una T4 con 4096 tokens

POR_CELDA = 10  # no cambiar
MUESTRAS = 5  # no cambiar

LIMITE_HORAS = 11.0

# ------------------- No hace falta editar desde aquí -------------------
import threading

DIR_SIN_THINK = RESULTADOS / "linea_base_sin_think"
DIR_CON_THINK = RESULTADOS / "linea_base_qwen06b"
LIMITE = INICIO + LIMITE_HORAS * 3600
SECO = ["--dry-run"] if PRUEBA_SECA else []


def linea_base(modelos, thinking, batch, carpeta):
    return [
        "scripts/run_pilot.py", "--stage", "piloto", "--models", *modelos, "--thinking", thinking,
        "--n-per-cell", POR_CELDA, "--n-samples", MUESTRAS,
        "--batch-size", batch, "--out", carpeta, *SECO,
    ]


SIN_THINK = linea_base(MODELOS_SIN_THINK, "off", BATCH_SIN_THINK, DIR_SIN_THINK)
CON_THINK = linea_base(MODELOS_CON_THINK, "on", BATCH_CON_THINK, DIR_CON_THINK)


def lanzar(nombre, argumentos, gpu=None):
    """Inicia un script del proyecto y muestra su salida, con el nombre de la tanda."""
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    proceso = subprocess.Popen(
        [sys.executable, "-u", *map(str, argumentos)], cwd=CODIGO, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )

    def leer():
        for linea in proceso.stdout:
            print(f"[{nombre}] {linea}", end="", flush=True)

    hilo = threading.Thread(target=leer, daemon=True)
    hilo.start()
    return nombre, proceso, hilo


def esperar(*tandas):
    """Espera a las tandas; si se pasa LIMITE_HORAS, las detiene (lo hecho queda guardado)."""
    while any(proceso.poll() is None for _, proceso, _ in tandas):
        if time.time() > LIMITE:
            print(f"Se alcanzo LIMITE_HORAS={LIMITE_HORAS}: se detiene para guardar.", flush=True)
            for _, proceso, _ in tandas:
                if proceso.poll() is None:
                    proceso.terminate()
            break
        time.sleep(5 if PRUEBA_SECA else 30)
    for nombre, proceso, hilo in tandas:
        try:
            proceso.wait(timeout=120)
        except subprocess.TimeoutExpired:
            proceso.kill()
            proceso.wait()
        hilo.join(timeout=10)
        print(f"[{nombre}] termino con codigo {proceso.returncode}", flush=True)


print(f"Se detiene a las {LIMITE_HORAS} h de iniciado el cuaderno.")

## 5. Correr

Primero descarga los modelos, para que las dos tandas no los bajen a la vez. Con dos GPU corren en paralelo; con una, la tanda sin thinking va primero. Cada ~25 respuestas se imprime el avance.


In [ ]:
if not PRUEBA_SECA:
    from huggingface_hub import snapshot_download

    for modelo in MODELOS_SIN_THINK + MODELOS_CON_THINK:
        snapshot_download(modelo)

if N_GPU >= 2:
    print("Dos GPU: sin thinking en la 0 y con thinking en la 1, en paralelo.")
    esperar(lanzar("sin think", SIN_THINK, gpu=0), lanzar("con think", CON_THINK, gpu=1))
else:
    print("Una GPU: primero sin thinking, despues con thinking.")
    esperar(lanzar("sin think", SIN_THINK))
    if time.time() < LIMITE - 1800:
        esperar(lanzar("con think", CON_THINK))
    else:
        print("No queda tiempo para la tanda con thinking en esta version.")
print(f"Tiempo total: {(time.time() - INICIO) / 3600:.2f} h")


## 6. Informes y zip

Corre aunque alguna tanda haya quedado incompleta: los informes se arman con lo que haya.


In [ ]:
from IPython.display import Markdown, display


def correr(*argumentos):
    resultado = subprocess.run(
        [sys.executable, "-u", *map(str, argumentos)], cwd=CODIGO, capture_output=True, text=True
    )
    print(resultado.stdout[-3000:], resultado.stderr[-3000:])
    return resultado.returncode


for carpeta in (DIR_SIN_THINK, DIR_CON_THINK):
    if (carpeta / "manifest.json").exists():
        print(f"{carpeta.name}:", rollouts_hechos(carpeta), "rollouts")
        correr("scripts/analyze_results.py", carpeta)
        reporte = carpeta / "report.md"
        if reporte.exists():
            display(Markdown(reporte.read_text(encoding="utf-8")))


In [ ]:
import datetime

ZIP_RESULTADOS = SALIDA / "resultados_otros_modelos.zip"
carpetas = [d for d in (DIR_SIN_THINK, DIR_CON_THINK) if d.exists()]
with zipfile.ZipFile(ZIP_RESULTADOS, "w", zipfile.ZIP_DEFLATED) as archivo:
    for carpeta in carpetas:
        for ruta in sorted(carpeta.iterdir()):
            if ruta.is_file() and not ruta.name.endswith(".tmp"):
                archivo.write(ruta, f"{carpeta.name}/{ruta.name}")
print(datetime.datetime.now().isoformat(timespec="seconds"))
print(f"LISTO ({ZIP_RESULTADOS.stat().st_size / 2**20:.1f} MiB): {ZIP_RESULTADOS}")
print("Descargalo desde la pestana Output de esta version.")